# 04 — Detection vs Attribution (proposal 3.3.4) → RQ1

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # so `import src...` works from notebooks/

import numpy as np
import pandas as pd

from src import config
from src.utils import set_seed, save_fig
set_seed()  # fix all RNGs -- reproducibility

Collapse all 11 LLMs into one `ai` class, rerun the SAME pipeline binary, compare against 12-way to quantify the attribution difficulty gap.

In [ ]:
from src import data, features, modeling
clean = data.load_or_build_clean(); splits = data.load_or_build_splits(clean)
y12 = clean[config.LABEL_COL].values
y_bin = np.where(y12 == config.HUMAN_CLASS, 'human', 'ai')

In [ ]:
# Best feature set = exp6_all (full model) + logreg (proposal's primary
# classifier) -- same condition, only the label vector changes.
texts = clean['text']
X_tfidf, _ = features.build_tfidf(texts.iloc[splits['train']], texts)
sty = features.build_stylometric(texts)
sty_scaled, _ = features.scale_dense(sty.values, splits['train'])
bib = features.build_biber(texts)
bib_scaled, _ = features.scale_dense(bib.values, splits['train'])
emb = features.build_sbert(texts)   # cached

blocks = {
    'tfidf': X_tfidf, 'stylometric': sty_scaled, 'biber': bib_scaled,
    'sbert': emb, 'length': clean[['log_token_count']].values,
}
X = features.assemble(blocks, features.EXPERIMENTS['exp6_all'], splits['train'])
Xtr, Xval = X[splits['train']], X[splits['val']]
ytr12, yval12 = y12[splits['train']], y12[splits['val']]
ytr_bin, yval_bin = y_bin[splits['train']], y_bin[splits['val']]

res_12way = modeling.train_and_evaluate('logreg', Xtr, ytr12, Xval, yval12)
res_binary = modeling.train_and_evaluate('logreg', Xtr, ytr_bin, Xval, yval_bin)
gap = res_binary.macro_f1 - res_12way.macro_f1

print(f"Binary Macro-F1: {res_binary.macro_f1:.4f}")
print(f"12-way Macro-F1: {res_12way.macro_f1:.4f}")
print(f"Attribution gap: {gap:.4f}")

In [ ]:
from src import viz

pd.DataFrame({
    'binary': {'macro_f1': res_binary.macro_f1, 'accuracy': res_binary.accuracy},
    '12way': {'macro_f1': res_12way.macro_f1, 'accuracy': res_12way.accuracy},
}).T.to_csv(config.ARTIFACTS / 'detection_vs_attribution.csv')

fig, ax = viz.new_fig(figsize=(4.5, 5))
labels = ['Binary\n(human vs AI)', '12-way\n(attribution)']
vals = [res_binary.macro_f1, res_12way.macro_f1]
bars = ax.bar(labels, vals, color=viz.CATEGORICAL[:2], width=0.55, zorder=3)
ax.bar_label(bars, fmt='%.3f', color=viz.INK_SECONDARY, padding=3)
ax.set_ylabel('Macro-F1')
ax.set_ylim(0, 1)
ax.set_title(f'Detection vs. attribution (gap = {gap:.3f})')
fig.tight_layout()
save_fig(fig, 'detection_vs_attribution')